Stock Market Performance & Investment Trends Analysis

Project objective

To analyse historical stock market performance, investment returns, trading activity, market volatility and dividend trends among selected publicly traded companies.

Data source: EOD Historical Data (EODHD)

Tools: Python, Pandas, SQL Server and Power BI.

Companies: Apple, Microsoft, Amazon, NVIDIA and Alphabet.

Initial data collection: Daily historical stock prices.

In [4]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from getpass import getpass

In [6]:
API_KEY = getpass.getpass("Enter your EODHD API key")

6aeefacfa9ea85.27512437 ········


In [7]:
if API_KEY:
    print('API key successfully captured')
else:
    print('API key is missing')

API key successfully captured


In [8]:
url = "https://eodhd.com/api/eod/AAPL.US"

params = {
    "api_token": API_KEY,
    "fmt": "json"
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [12]:
# the API response into JSON
data = response.json()

# the type of data returned
print(type(data))

# the first record
print(data[0])

<class 'list'>
{'date': '2025-11-07', 'open': 269.795, 'high': 272.29, 'low': 266.77, 'close': 268.47, 'adjusted_close': 267.4819, 'volume': 48227365}


In [13]:
# Converting the JSON response into a DataFrame
df = pd.DataFrame(data)

# the first five records
display(df.head())

# the number of rows and columns
print("Dataset shape:", df.shape)

# column names and data types
df.info()

,date,open,high,low,close,adjusted_close,volume
0,2025-11-07,269.795,272.290,266.770,268.47,267.4819,48227365
1,2025-11-10,268.960,273.730,267.455,269.43,268.6986,41312412
2,2025-11-11,269.810,275.910,269.800,275.25,274.5028,46208318
3,2025-11-12,275.000,275.730,271.700,273.47,272.7276,48397982
4,2025-11-13,274.110,276.699,272.090,272.95,272.2091,49602794


Dataset shape: (251, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            251 non-null    object 
 1   open            251 non-null    float64
 2   high            251 non-null    float64
 3   low             251 non-null    float64
 4   close           251 non-null    float64
 5   adjusted_close  251 non-null    float64
 6   volume          251 non-null    int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 13.9+ KB


In [14]:
# Checking the earliest and latest trading dates
print("Earliest date:", df["date"].min())
print("Latest date:", df["date"].max())

# Converting date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Checking the data types again
print("\nUpdated data types:")
print(df.dtypes)

Earliest date: 2025-11-07
Latest date: 2026-11-06

Updated data types:
date              datetime64[ns]
open                     float64
high                     float64
low                      float64
close                    float64
adjusted_close           float64
volume                     int64
dtype: object


In [15]:
# Companies analysed
companies = {
    "Apple": "AAPL.US",
    "Microsoft": "MSFT.US",
    "Amazon": "AMZN.US",
    "NVIDIA": "NVDA.US",
    "Alphabet": "GOOGL.US"
}

# Historical date range
start_date = "2025-11-07"
end_date = "2026-11-06"

print("Companies selected:", len(companies))
print(companies)

Companies selected: 5
{'Apple': 'AAPL.US', 'Microsoft': 'MSFT.US', 'Amazon': 'AMZN.US', 'NVIDIA': 'NVDA.US', 'Alphabet': 'GOOGL.US'}


In [16]:

# EXTRACTING STOCK DATA FOR ALL FIVE COMPANIES


stock_data = {}

for company, ticker in companies.items():

    print(f"\nRetrieving {company} ({ticker})...")

    url = f"https://eodhd.com/api/eod/{ticker}"

    params = {
        "api_token": API_KEY,
        "from": start_date,
        "to": end_date,
        "fmt": "json"
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        company_data = response.json()

        if not isinstance(company_data, list) or not company_data:
            print(f"No usable data returned for {company}.")
            continue

        company_df = pd.DataFrame(company_data)

        company_df["company"] = company
        company_df["ticker"] = ticker

        stock_data[company] = company_df

        print(f"Successfully retrieved {len(company_df)} records.")

    except requests.exceptions.RequestException as error:
        print(f"Request failed: {error}")

    except ValueError:
        print("The API response could not be interpreted as JSON.")



# COMBINING ALL FIVE COMPANIES


if len(stock_data) == len(companies):

    all_stocks = pd.concat(stock_data.values(), ignore_index=True)

    print("\nEXTRACTION SUMMARY")
    print("------------------")
    print("Companies retrieved:", all_stocks["company"].nunique())
    print("Total records:", len(all_stocks))
    print(all_stocks["company"].value_counts())

else:
    print(f"\nOnly {len(stock_data)} out of {len(companies)} companies retrieved.")


Retrieving Apple (AAPL.US)...
Successfully retrieved 251 records.

Retrieving Microsoft (MSFT.US)...
Successfully retrieved 251 records.

Retrieving Amazon (AMZN.US)...
Successfully retrieved 251 records.

Retrieving NVIDIA (NVDA.US)...
Successfully retrieved 251 records.

Retrieving Alphabet (GOOGL.US)...
Successfully retrieved 251 records.

EXTRACTION SUMMARY
------------------
Companies retrieved: 5
Total records: 1255
company
Apple        251
Microsoft    251
Amazon       251
NVIDIA       251
Alphabet     251
Name: count, dtype: int64


In [17]:
# DATA VALIDATION

# 1. Inspect the first five records
display(all_stocks.head())

# 2. Check dataset dimensions
print("Dataset shape:", all_stocks.shape)

# 3. Check missing values
print("\nMissing values:")
print(all_stocks.isnull().sum())

# 4. Check duplicate records
print("\nDuplicate records:", all_stocks.duplicated(
    subset=["date", "ticker"]
).sum())

# 5. Check date coverage per company
print("\nDate coverage:")
print(
    all_stocks.groupby("company")["date"]
    .agg(["min", "max", "count"])
)

# 6. Check data types
print("\nData types:")
print(all_stocks.dtypes)

,date,open,high,low,close,adjusted_close,volume,company,ticker
0,2025-11-07,269.795,272.290,266.770,268.47,267.4819,48227365,Apple,AAPL.US
1,2025-11-10,268.960,273.730,267.455,269.43,268.6986,41312412,Apple,AAPL.US
2,2025-11-11,269.810,275.910,269.800,275.25,274.5028,46208318,Apple,AAPL.US
3,2025-11-12,275.000,275.730,271.700,273.47,272.7276,48397982,Apple,AAPL.US
4,2025-11-13,274.110,276.699,272.090,272.95,272.2091,49602794,Apple,AAPL.US


Dataset shape: (1255, 9)

Missing values:
date              0
open              0
high              0
low               0
close             0
adjusted_close    0
volume            0
company           0
ticker            0
dtype: int64

Duplicate records: 0

Date coverage:
                  min         max  count
company                                 
Alphabet   2025-11-07  2026-11-06    251
Amazon     2025-11-07  2026-11-06    251
Apple      2025-11-07  2026-11-06    251
Microsoft  2025-11-07  2026-11-06    251
NVIDIA     2025-11-07  2026-11-06    251

Data types:
date               object
open              float64
high              float64
low               float64
close             float64
adjusted_close    float64
volume              int64
company            object
ticker             object
dtype: object


In [18]:
#STANDARDISE DATA TYPES

# Converting date to datetime
all_stocks["date"] = pd.to_datetime(all_stocks["date"])

# Checking columns data types
price_columns = [
    "open",
    "high",
    "low",
    "close",
    "adjusted_close"
]

all_stocks[price_columns] = all_stocks[price_columns].astype(float)

all_stocks["volume"] = all_stocks["volume"].astype("int64")

# Verifying changes
print(all_stocks.dtypes)

date              datetime64[ns]
open                     float64
high                     float64
low                      float64
close                    float64
adjusted_close           float64
volume                     int64
company                   object
ticker                    object
dtype: object


In [20]:
# Saving the combined dataset
all_stocks.to_csv(
    r'D:\PORTIFOLIO PROJECTS\Stock_Market_Investment_Analysis\Data\stock_market_data.csv',
    index=False
)

print("Dataset successfully saved!")

Dataset successfully saved!


In [22]:
# PRICE CONSISTENCY


# Check whether daily high is below open or close
invalid_high = all_stocks[
    (all_stocks["high"] < all_stocks["open"]) |
    (all_stocks["high"] < all_stocks["close"])
]

# Check whether daily low is above open or close
invalid_low = all_stocks[
    (all_stocks["low"] > all_stocks["open"]) |
    (all_stocks["low"] > all_stocks["close"])
]

# Check for zero or negative prices
invalid_prices = all_stocks[
    (all_stocks[["open", "high", "low", "close",
                 "adjusted_close"]] <= 0).any(axis=1)
]

print("Invalid high prices:", len(invalid_high))
print("Invalid low prices:", len(invalid_low))
print("Zero or negative prices:", len(invalid_prices))

Invalid high prices: 0
Invalid low prices: 0
Zero or negative prices: 0


In [23]:
# COMPARISON CLOSING AND ADJUSTED PRICES

# the difference between closing prices
all_stocks["adjustment_difference"] = (
    all_stocks["adjusted_close"] - all_stocks["close"]
)

# Summarise the differences by company
adjustment_summary = (
    all_stocks.groupby("company")["adjustment_difference"]
    .agg(["min", "max", "mean"])
    .round(4)
)

display(adjustment_summary)

,min,max,mean
company,,,
Alphabet,-0.8390,0.0,-0.3749
Amazon,0.0000,0.0,0.0000
Apple,-0.9881,0.0,-0.3650
Microsoft,-4.1712,0.0,-1.4487
NVIDIA,-0.5376,0.0,-0.3125


In [24]:
# DAILY STOCK RETURNS

# Sort data by company and trading date
all_stocks = all_stocks.sort_values(
    by=["ticker", "date"]
)

# Calculate daily percentage returns
all_stocks["daily_return"] = (
    all_stocks.groupby("ticker")["adjusted_close"]
    .pct_change() * 100
)

# Inspect the results
display(
    all_stocks[
        ["date", "company", "adjusted_close", "daily_return"]
    ].head(10)
)

,date,company,adjusted_close,daily_return
0,2025-11-07,Apple,267.4819,NaN
1,2025-11-10,Apple,268.6986,0.454872
2,2025-11-11,Apple,274.5028,2.160115
3,2025-11-12,Apple,272.7276,-0.646696
4,2025-11-13,Apple,272.2091,-0.190116
5,2025-11-14,Apple,271.6705,-0.197863
6,2025-11-17,Apple,266.7340,-1.817091
7,2025-11-18,Apple,266.7140,-0.007498
8,2025-11-19,Apple,267.8310,0.418801
9,2025-11-20,Apple,265.5272,-0.860169


In [25]:
#  CUMULATIVE RETURNS

# Calculate cumulative growth for each company
all_stocks["cumulative_return"] = (
    all_stocks.groupby("ticker")["daily_return"]
    .transform(lambda x: ((1 + x / 100).cumprod() - 1) * 100)
)

# Inspect the results
display(
    all_stocks[
        ["date", "company", "adjusted_close",
         "daily_return", "cumulative_return"]
    ].groupby("company").tail(3)
)

,date,company,adjusted_close,daily_return,cumulative_return
248,2026-11-04,Apple,349.51,5.305815,30.666785
249,2026-11-05,Apple,344.17,-1.527853,28.670389
250,2026-11-06,Apple,342.37,-0.522997,27.997446
750,2026-11-04,Amazon,266.03,6.119111,8.845792
751,2026-11-05,Amazon,260.21,-2.187723,6.464547
752,2026-11-06,Amazon,259.30,-0.349718,6.092222
1252,2026-11-04,Alphabet,288.93,3.178231,3.891780
1253,2026-11-05,Alphabet,282.57,-2.201225,1.604888
1254,2026-11-06,Alphabet,284.78,0.782107,2.399547
499,2026-11-04,Microsoft,555.80,6.681510,12.791941


In [26]:
#  RANK STOCK PERFORMANCE

# Get the latest cumulative return for each company
performance_summary = (
    all_stocks.sort_values("date")
    .groupby("company")
    .tail(1)
    [["company", "adjusted_close", "cumulative_return"]]
    .sort_values("cumulative_return", ascending=False)
    .reset_index(drop=True)
)

# Display the ranking
display(performance_summary)

,company,adjusted_close,cumulative_return
0,Apple,342.37,27.997446
1,Microsoft,530.61,7.679979
2,Amazon,259.30,6.092222
3,Alphabet,284.78,2.399547
4,NVIDIA,186.64,-0.564890


In [27]:
#  INVESTMENT RISK ANALYSIS

risk_summary = (
    all_stocks.groupby("company")["daily_return"]
    .agg(
        average_daily_return="mean",
        daily_volatility="std",
        best_day="max",
        worst_day="min"
    )
    .reset_index()
)

# Round results to four decimal places
risk_summary = risk_summary.round(4)

# Rank companies by daily volatility
risk_summary = risk_summary.sort_values(
    "daily_volatility",
    ascending=False
)

display(risk_summary)

,company,average_daily_return,daily_volatility,best_day,worst_day
4,NVIDIA,0.0269,2.4206,8.7379,-9.2590
1,Amazon,0.0460,2.1393,15.3206,-5.5548
3,Microsoft,0.0519,2.1338,15.5067,-9.9932
0,Alphabet,0.0322,2.1306,9.9617,-12.6412
2,Apple,0.1111,1.5677,5.3058,-7.3539


In [28]:
# ANNUALISED VOLATILITY

import numpy as np

risk_summary["annualised_volatility"] = (
    risk_summary["daily_volatility"] * np.sqrt(252)
)

# Convert to percentage and round
risk_summary["annualised_volatility"] = (
    risk_summary["annualised_volatility"].round(2)
)

# Display annualised risk ranking
display(
    risk_summary[
        ["company", "daily_volatility", "annualised_volatility"]
    ].sort_values(
        "annualised_volatility",
        ascending=False
    )
)

,company,daily_volatility,annualised_volatility
4,NVIDIA,2.4206,38.43
1,Amazon,2.1393,33.96
3,Microsoft,2.1338,33.87
0,Alphabet,2.1306,33.82
2,Apple,1.5677,24.89


In [29]:
# MAXIMUM DRAWDOWN ANALYSIS

# Calculating running peak adjusted closing price
all_stocks["running_peak"] = (
    all_stocks.groupby("ticker")["adjusted_close"]
    .cummax()
)

# Calculating drawdown percentage
all_stocks["drawdown"] = (
    (all_stocks["adjusted_close"] /
     all_stocks["running_peak"] - 1) * 100
)

# Find maximum drawdown for each company
drawdown_summary = (
    all_stocks.groupby("company")["drawdown"]
    .min()
    .reset_index(name="maximum_drawdown")
)

# Sort from worst drawdown to best
drawdown_summary = drawdown_summary.sort_values(
    "maximum_drawdown"
).round(2)

display(drawdown_summary)

,company,maximum_drawdown
0,Alphabet,-30.62
3,Microsoft,-30.53
4,NVIDIA,-24.09
1,Amazon,-20.20
2,Apple,-13.80


In [30]:
# RISK-ADJUSTED PERFORMANCE

# Calculating annualised Sharpe ratio
risk_summary["sharpe_ratio"] = (
    risk_summary["average_daily_return"] /
    risk_summary["daily_volatility"]
) * np.sqrt(252)

# Rounding results
risk_summary["sharpe_ratio"] = (
    risk_summary["sharpe_ratio"].round(3)
)

# Ranking companies by Sharpe ratio
sharpe_summary = risk_summary[
    ["company", "average_daily_return",
     "annualised_volatility", "sharpe_ratio"]
].sort_values(
    "sharpe_ratio",
    ascending=False
)

display(sharpe_summary)

,company,average_daily_return,annualised_volatility,sharpe_ratio
2,Apple,0.1111,24.89,1.125
3,Microsoft,0.0519,33.87,0.386
1,Amazon,0.0460,33.96,0.341
0,Alphabet,0.0322,33.82,0.240
4,NVIDIA,0.0269,38.43,0.176


In [35]:
#Performance summary
performance_summary = (
    all_stocks.sort_values("date")
    .groupby("company")
    .tail(1)
    [["company", "ticker", "adjusted_close", "cumulative_return"]]
    .sort_values("cumulative_return", ascending=False)
    .reset_index(drop=True)
)


In [38]:
#Risk summary
risk_summary.to_csv(
    r'D:\PORTIFOLIO PROJECTS\Stock_Market_Investment_Analysis\Data\stock_risk_summary.csv',
    index=False
)


In [39]:
# Drawdown summary
drawdown_summary.to_csv(
    r'D:\PORTIFOLIO PROJECTS\Stock_Market_Investment_Analysis\Data\stock_drawdown_summary.csv',
    index=False
)


In [40]:
# Performance summary
performance_summary.to_csv(
    r'D:\PORTIFOLIO PROJECTS\Stock_Market_Investment_Analysis\Data\stock_performance_summary.csv',
    index=False
)

In [42]:
all_stocks.to_csv(
    r'D:\PORTIFOLIO PROJECTS\Stock_Market_Investment_Analysis\Data\stock_market_data.csv',
    index=False)

In [43]:
# ==========================================
# FINAL DATASET VALIDATION
# ==========================================

import os
import pandas as pd

# Folder containing our exported datasets
data_folder = r'D:\PORTIFOLIO PROJECTS\Stock_Market_Investment_Analysis\Data'

# List of exported files
files = [
    "stock_market_data.csv",
    "stock_performance_summary.csv",
    "stock_risk_summary.csv",
    "stock_drawdown_summary.csv"
]

# Check each file
for file in files:

    file_path = os.path.join(data_folder, file)

    if os.path.exists(file_path):

        exported_data = pd.read_csv(file_path)

        print(f"\nFile: {file}")
        print("Status: Available")
        print("Rows:", exported_data.shape[0])
        print("Columns:", exported_data.shape[1])

    else:
        print(f"\nFile: {file}")
        print("Status: NOT FOUND")


File: stock_market_data.csv
Status: Available
Rows: 1255
Columns: 14

File: stock_performance_summary.csv
Status: Available
Rows: 5
Columns: 4

File: stock_risk_summary.csv
Status: Available
Rows: 5
Columns: 7

File: stock_drawdown_summary.csv
Status: Available
Rows: 5
Columns: 2
